# Demo 6：Equilibrium Characterization

任务：`equilibrium-characterization`。本 Demo 使用两次浓度干预和 pH/UV-vis 反馈表征隐藏酸碱、沉淀与非理想活度关系。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'equilibrium-characterization'
SEED = 0

## 1. 公开任务合同

Agent 只获得公开 pH-meter、UV-vis 与 final assay 观测，不读取 hidden state、pKa、Ksp、物种量或活度系数。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,equilibrium-characterization
1,motivation,Characterize a bounded aqueous-equilibrium sli...
2,budget,24
3,episode_mode,campaign
4,success_metrics,"[pH_normalized, acid_dissociation_fraction, pr..."


## 2. 构造候选干预

每个候选条件改变初始体积以及两次试剂加入量，从而生成一条短小但可辨识的响应序列。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.028..."
1,2,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
2,3,measure,"{'operation': 'measure', 'instrument': 'ph_met..."
3,4,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
4,5,measure,"{'operation': 'measure', 'instrument': 'ph_met..."
5,6,measure,"{'operation': 'measure', 'instrument': 'uvvis'}"
6,7,terminate,{'operation': 'terminate'}
7,8,measure,"{'operation': 'measure', 'instrument': 'final_..."


## 3. 执行并读取反馈

比较公开 `processed_estimate` 中的 `pH_normalized`、`acid_dissociation_fraction`、`precipitation_signal`、`equilibrium_residual` 和 `equilibrium_confidence`。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'pH_normalized', 'acid_dissociation_fraction', 'precipitation_signal', 'equilibrium_residual', 'equilibrium_confidence', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,pH_normalized,acid_dissociation_fraction,precipitation_signal,equilibrium_residual,equilibrium_confidence,all_actions_valid
0,low,0.269977,0.281465,0.078391,0.159061,0.001058,0.450128,True
1,mid,0.233940,0.281406,0.078266,0.159072,0.001058,0.370110,True
2,high,0.233909,0.281388,0.078228,0.159075,0.001058,0.370061,True


### 查看两次浓度探针后的反馈

响应序列比单个终点更适合判断局部平衡 world model。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,3,measure,0.292957,NaN,"cost, safety_risk, score, pH_normalized, acid_...","{'pH_normalized': 0.2789731242333648, 'acid_di..."
1,5,measure,-0.062639,NaN,"cost, safety_risk, score, pH_normalized, acid_...","{'pH_normalized': 0.283594959668456, 'acid_dis..."
2,6,measure,-0.230319,NaN,"cost, safety_risk, score",{}
3,8,measure,0.233940,0.23394,"cost, safety_risk, score, pH_normalized, acid_...","{'pH_normalized': 0.28140570112994545, 'acid_d..."


## 4. 同一干预，不同隐藏规律

World B 使用 `constitutive_law_family` 教学控制改变隐藏活度系数关系。公共任务语义保持不变，Agent 必须从响应曲线发现理想模型不再充分。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='constitutive_law_family', seed=SEED
)
paired_worlds

,opaque_world,pH_normalized,acid_dissociation_fraction,precipitation_signal,equilibrium_residual,equilibrium_confidence,leaderboard_score,cost,safety_risk,all_actions_valid
0,World A,0.281406,0.078266,0.159072,0.001058,0.370110,0.233940,0.0,None,True
1,World B,0.325802,0.078121,0.159084,0.001058,0.416611,0.259277,0.0,None,True


## 5. 留给 Agent 的能力问题

机制识别可能不可唯一，因此优先评估未见干预预测与不确定性，而不是要求逐字匹配隐藏方程。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行浓度条件下的 pH、解离和沉淀响应及不确定性。',
    'next_intervention_query': '选择最能区分理想与非理想平衡解释的下一次浓度探针。',
    'evaluation_note': '允许等价机制或 abstention，主评测采用 held-out intervention prediction。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'pH_normalized': 0.28146522333141655,
   'acid_dissociation_fraction': 0.07839120304194294,
   'precipitation_signal': 0.1590613250019027,
   'equilibrium_residual': 0.001057822521317429,
   'equilibrium_confidence': 0.45012758821048104,
   'leaderboard_score': 0.2699769207562028,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 8},
  {'candidate': 'mid',
   'pH_normalized': 0.28140570112994545,
   'acid_dissociation_fraction': 0.0782658733721205,
   'precipitation_signal': 0.1590721551085391,
   'equilibrium_residual': 0.0010578225213172196,
   'equilibrium_confidence': 0.37011034685430166,
   'leaderboard_score': 0.23393976850780596,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 8},
  {'candidate': 'high',
   'pH_normalized': 0.28138769884146053,
   'acid_dissociation_fraction': 0.07822800853235674,
   'precipitation_signal': 0.15907534974754267,
   '